# AutoARIMAX Measles Cumulative Forecast

This notebook fits one model only: **AutoARIMAX** on `log1p(annual_cases)`, then reconstructs cumulative measles cases from nonnegative annual increments. This prevents impossible decreases. The exogenous variables retain `time`, `outbreak_2014`, `outbreak_2019`, and `outbreak_2025`; outbreak years are encoded as one-year pulses so their effects are not incorrectly carried into later years. The historical 95% band is generated from simulated cumulative paths using centered residuals on the original case scale. This avoids the incorrect practice of summing marginal annual confidence-limit endpoints, which made the cumulative interval explode. The 2026 interval is calibrated from rolling one-year-ahead errors for 2016-2025 and constrained by the cases already observed through **May 1, 2026**.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pmdarima import auto_arima

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = Path("data-table (1).csv")
TRAIN_START_YEAR = 1996
FORECAST_COUNT_YEAR = 2026
PARTIAL_2026_DATE = pd.Timestamp("2026-05-01")
FULL_YEAR_2026_FORECAST_DATE = pd.Timestamp("2026-12-31")
CALIBRATION_START_YEAR = 2016
N_CI_SIMULATIONS = 20_000
RANDOM_SEED = 20260608


In [2]:
def load_cumulative_data(path: Path) -> pd.DataFrame:
    data = pd.read_csv(path)
    if data.columns[0].lower().startswith("unnamed") or data.columns[0] not in {"year", "cases"}:
        data = data.drop(columns=data.columns[0])

    data["year"] = data["year"].astype(int)
    data["cases"] = data["cases"].astype(float)
    data = data[["year", "cases"]].dropna().sort_values("year").reset_index(drop=True)
    data["cumulative_cases"] = data["cases"].cumsum()
    data["plot_date"] = pd.to_datetime({"year": data["year"] + 1, "month": 1, "day": 1})
    data.loc[data["year"] == FORECAST_COUNT_YEAR, "plot_date"] = PARTIAL_2026_DATE
    return data.rename(columns={"year": "count_year"}).set_index("plot_date").rename_axis("date")


def create_features(df: pd.DataFrame, base_year: int) -> pd.DataFrame:
    df = df.copy()
    years = df["count_year"]
    df["time"] = years - base_year
    df["outbreak_2014"] = (years == 2014).astype(int)
    df["outbreak_2019"] = (years == 2019).astype(int)
    df["outbreak_2025"] = (years == 2025).astype(int)
    return df


def fit_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    return {
        "n_scored": int(valid.sum()),
        "mae": mean_absolute_error(y_true[valid], y_pred[valid]),
        "rmse": np.sqrt(mean_squared_error(y_true[valid], y_pred[valid])),
    }


all_data = load_cumulative_data(DATA_PATH)
train = all_data[(all_data["count_year"] >= TRAIN_START_YEAR) & (all_data["count_year"] < FORECAST_COUNT_YEAR)].copy()
partial_2026 = all_data[all_data["count_year"] == FORECAST_COUNT_YEAR].copy()

base_year = int(train["count_year"].min())
train_features = create_features(train, base_year)
future_features = create_features(
    pd.DataFrame({"count_year": [FORECAST_COUNT_YEAR]}, index=[FULL_YEAR_2026_FORECAST_DATE]),
    base_year,
)

exog_cols = ["time", "outbreak_2014", "outbreak_2019", "outbreak_2025"]
y = train_features["cumulative_cases"].astype(float)
annual_cases = train_features["cases"].astype(float)
annual_cases_log = np.log1p(annual_cases)
X_train = train_features[exog_cols].astype(float)
X_future = future_features[exog_cols].astype(float)


In [3]:
def fit_autoarimax(y_log, X):
    return auto_arima(
        y_log,
        X=X,
        seasonal=False,
        stepwise=True,
        max_d=1,
        suppress_warnings=True,
        error_action="ignore",
        information_criterion="aic",
        start_p=0,
        max_p=1,
        start_q=0,
        max_q=5,
        start_P=0,
        max_P=0,
        start_Q=0,
        max_Q=0,
    )


arimax_model = fit_autoarimax(annual_cases_log, X_train)

burn_in = int(arimax_model.order[1])
fitted_annual_cases = pd.Series(np.nan, index=y.index, dtype=float)

if burn_in > 0:
    fitted_annual_cases.iloc[:burn_in] = annual_cases.iloc[:burn_in]
fitted_annual_cases.iloc[burn_in:] = np.maximum(
    np.expm1(arimax_model.predict_in_sample(start=burn_in, end=len(y) - 1, X=X_train)),
    0.0,
)

# Simulate full cumulative paths instead of summing annual CI endpoints.
cumulative_before_training = float(y.iloc[0] - annual_cases.iloc[0])
fitted_cumulative_cases = cumulative_before_training + fitted_annual_cases.cumsum()
case_residuals = np.asarray(annual_cases - fitted_annual_cases, dtype=float)
case_residuals = case_residuals - case_residuals.mean()
rng = np.random.default_rng(RANDOM_SEED)
simulated_residuals = rng.choice(
    case_residuals,
    size=(N_CI_SIMULATIONS, len(annual_cases)),
    replace=True,
)
simulated_annual_cases = np.maximum(
    np.asarray(fitted_annual_cases, dtype=float)[None, :] + simulated_residuals,
    0.0,
)
simulated_cumulative_cases = cumulative_before_training + np.cumsum(
    simulated_annual_cases, axis=1
)
fitted_ci_lower, fitted_ci_upper = np.quantile(
    simulated_cumulative_cases, [0.025, 0.975], axis=0
)

forecast_log, forecast_ci_log = arimax_model.predict(
    n_periods=1, X=X_future, return_conf_int=True, alpha=0.05
)
partial_2026_cumulative = float(partial_2026["cumulative_cases"].iloc[0]) if not partial_2026.empty else np.nan
cumulative_through_2025 = float(all_data.loc[all_data["count_year"] < FORECAST_COUNT_YEAR, "cases"].sum())
raw_forecast_2026_cases_array = np.maximum(
    np.expm1(np.asarray(forecast_log, dtype=float)), 0.0
)
parametric_forecast_2026_cases_ci = np.maximum(
    np.expm1(np.asarray(forecast_ci_log, dtype=float)), 0.0
)

# Calibrate the prediction interval with genuine one-year-ahead errors.
rolling_errors = []
for test_year in range(CALIBRATION_START_YEAR, FORECAST_COUNT_YEAR):
    fold_mask = train_features["count_year"] < test_year
    fold_model = fit_autoarimax(
        annual_cases_log.loc[fold_mask], X_train.loc[fold_mask]
    )
    fold_future = create_features(
        pd.DataFrame({"count_year": [test_year]}), base_year
    )[exog_cols].astype(float)
    fold_forecast_log = fold_model.predict(n_periods=1, X=fold_future)
    fold_forecast_cases = float(
        np.maximum(np.expm1(np.asarray(fold_forecast_log, dtype=float)), 0.0)[0]
    )
    observed_cases = float(
        train_features.loc[train_features["count_year"] == test_year, "cases"].iloc[0]
    )
    rolling_errors.append(abs(observed_cases - fold_forecast_cases))

calibration_error_95 = float(
    np.quantile(rolling_errors, 0.95, method="higher")
)
calibrated_forecast_2026_cases_ci = np.array([
    [
        max(0.0, raw_forecast_2026_cases_array[0] - calibration_error_95),
        raw_forecast_2026_cases_array[0] + calibration_error_95,
    ]
])
raw_forecast_cumulative_cases = cumulative_through_2025 + raw_forecast_2026_cases_array
raw_forecast_ci = cumulative_through_2025 + calibrated_forecast_2026_cases_ci
forecast_cumulative_cases = raw_forecast_cumulative_cases.copy()
forecast_ci = raw_forecast_ci.copy()
if np.isfinite(partial_2026_cumulative):
    forecast_cumulative_cases[0] = max(forecast_cumulative_cases[0], partial_2026_cumulative)
    forecast_ci[0, 0] = max(forecast_ci[0, 0], partial_2026_cumulative)
    forecast_ci[0, 1] = max(forecast_ci[0, 1], forecast_cumulative_cases[0])
forecast_2026_cases = float(forecast_cumulative_cases[0] - cumulative_through_2025)
raw_forecast_2026_cases = float(raw_forecast_2026_cases_array[0])

fitted_values = pd.DataFrame(
    {
        "count_year": train_features["count_year"],
        "observed_cumulative_cases": y,
        "observed_annual_cases": annual_cases,
        "autoarimax_fitted_annual_cases": fitted_annual_cases,
        "autoarimax_fitted_cumulative_cases": fitted_cumulative_cases,
        "ci_lower_95": fitted_ci_lower,
        "ci_upper_95": fitted_ci_upper,
    },
    index=train_features.index,
)

sarimax_results = arimax_model.arima_res_
param_uncertainty = pd.DataFrame(
    {
        "coef_log_scale": sarimax_results.params,
        "std_error": sarimax_results.bse,
        "z_value": sarimax_results.zvalues,
        "p_value": sarimax_results.pvalues,
    },
    index=sarimax_results.param_names,
)
conf_int = pd.DataFrame(
    np.asarray(sarimax_results.conf_int(alpha=0.05)),
    index=sarimax_results.param_names,
    columns=["ci_lower_95_log_scale", "ci_upper_95_log_scale"],
)
param_uncertainty = param_uncertainty.join(conf_int)
param_uncertainty["multiplicative_effect_on_one_plus_cases"] = np.exp(param_uncertainty["coef_log_scale"])
param_uncertainty["multiplicative_ci_lower_95"] = np.exp(param_uncertainty["ci_lower_95_log_scale"])
param_uncertainty["multiplicative_ci_upper_95"] = np.exp(param_uncertainty["ci_upper_95_log_scale"])

model_summary = pd.DataFrame([
    {
        "model": "AutoARIMAX annual increments to cumulative",
        "exogenous_variables": ", ".join(exog_cols),
        "forecast_count_year": FORECAST_COUNT_YEAR,
        "partial_2026_observed_date": PARTIAL_2026_DATE.date().isoformat(),
        "partial_2026_observed_cumulative_cases": partial_2026_cumulative,
        "full_year_2026_forecast_date": FULL_YEAR_2026_FORECAST_DATE.date().isoformat(),
        "raw_forecast_cumulative_cases": float(raw_forecast_cumulative_cases[0]),
        "forecast_cumulative_cases": float(forecast_cumulative_cases[0]),
        "raw_forecast_2026_cases_implied": raw_forecast_2026_cases,
        "forecast_2026_cases_implied": forecast_2026_cases,
        "historical_interval_method": "case-scale residual simulation of cumulative paths",
        "forecast_interval_method": "rolling absolute-error calibration, 2016-2025",
        "calibration_error_95_cases": calibration_error_95,
        "parametric_ci_lower_95_cases": float(parametric_forecast_2026_cases_ci[0, 0]),
        "parametric_ci_upper_95_cases": float(parametric_forecast_2026_cases_ci[0, 1]),
        "ci_lower_95": float(forecast_ci[0, 0]),
        "ci_upper_95": float(forecast_ci[0, 1]),
        "aic": float(arimax_model.aic()),
        "order": str(arimax_model.order),
        **fit_metrics(fitted_values["observed_cumulative_cases"], fitted_values["autoarimax_fitted_cumulative_cases"]),
    }
])

display(model_summary)
display(param_uncertainty)
display(fitted_values.tail())


,model,exogenous_variables,forecast_count_year,partial_2026_observed_date,partial_2026_observed_cumulative_cases,full_year_2026_forecast_date,raw_forecast_cumulative_cases,forecast_cumulative_cases,raw_forecast_2026_cases_implied,forecast_2026_cases_implied,...,calibration_error_95_cases,parametric_ci_lower_95_cases,parametric_ci_upper_95_cases,ci_lower_95,ci_upper_95,aic,order,n_scored,mae,rmse
0,AutoARIMAX annual increments to cumulative,"time, outbreak_2014, outbreak_2019, outbreak_2025",2026,2026-05-01,84565.0,2026-12-31,82839.465358,84565.0,88.465358,1814.0,...,2196.254279,20.899175,364.495514,84565.0,85035.719638,77.264959,"(0, 0, 0)",30,457.97341,499.214531


,coef_log_scale,std_error,z_value,p_value,ci_lower_95_log_scale,ci_upper_95_log_scale,multiplicative_effect_on_one_plus_cases,multiplicative_ci_lower_95,multiplicative_ci_upper_95
intercept,4.587476,0.276308,16.602777,6.653995e-62,4.045923,5.129029,98.246153,57.163915,168.853142
time,-0.003121,0.015075,-0.207020,8.359943e-01,-0.032667,0.026426,0.996884,0.967861,1.026778
outbreak_2014,1.973093,4907.290078,0.000402,9.996792e-01,-9616.138722,9620.084908,7.192888,0.000000,inf
outbreak_2019,2.635055,10287.180266,0.000256,9.997956e-01,-20159.867769,20165.137878,13.944076,0.000000,inf
outbreak_2025,3.237571,32598.521577,0.000099,9.999208e-01,-63888.690669,63895.165810,25.471765,0.000000,inf
sigma2,0.515633,0.134388,3.836887,1.246037e-04,0.252236,0.779029,1.674698,1.286900,2.179355


,count_year,observed_cumulative_cases,observed_annual_cases,autoarimax_fitted_annual_cases,autoarimax_fitted_cumulative_cases,ci_lower_95,ci_upper_95
date,,,,,,,
2022-01-01,2021,80001.0,49.0,89.872333,79339.158392,78423.015078,80460.060459
2023-01-01,2022,80122.0,121.0,89.589179,79428.747571,78495.971870,80569.437254
2024-01-01,2023,80181.0,59.0,89.306907,79518.054479,78570.399602,80680.829548
2025-01-01,2024,80466.0,285.0,89.025515,79607.079994,78634.472301,80786.920007
2026-01-01,2025,82751.0,2285.0,2284.963535,81892.043529,80900.760562,83095.724086


In [4]:
fig, ax = plt.subplots(figsize=(12, 6))

historical_complete = all_data[all_data["count_year"] < FORECAST_COUNT_YEAR].copy()
model_curve = pd.concat([
    fitted_values["autoarimax_fitted_cumulative_cases"],
    pd.Series(
        [partial_2026_cumulative, forecast_cumulative_cases[0]],
        index=[PARTIAL_2026_DATE, FULL_YEAR_2026_FORECAST_DATE],
        name="autoarimax_curve",
    ),
]).sort_index()
forecast_interval_dates = [PARTIAL_2026_DATE, FULL_YEAR_2026_FORECAST_DATE]
forecast_interval_lower = [partial_2026_cumulative, forecast_ci[0, 0]]
forecast_interval_upper = [partial_2026_cumulative, forecast_ci[0, 1]]

ax.plot(
    historical_complete.index,
    historical_complete["cumulative_cases"],
    marker="o",
    color="black",
    label="Observed cumulative cases",
)
ax.fill_between(
    fitted_values.index,
    fitted_values["ci_lower_95"],
    fitted_values["ci_upper_95"],
    color="tab:green",
    alpha=0.12,
    label="Historical simulated 95% prediction band",
)
ax.fill_between(
    forecast_interval_dates,
    forecast_interval_lower,
    forecast_interval_upper,
    color="tab:green",
    alpha=0.16,
    label="Calibrated 95% prediction interval",
)
ax.plot(
    model_curve.index,
    model_curve,
    color="tab:green",
    linestyle="--",
    linewidth=2.3,
    label="AutoARIMAX reconstructed cumulative curve",
)
ax.scatter(
    [PARTIAL_2026_DATE],
    [partial_2026_cumulative],
    color="tab:red",
    marker="x",
    s=110,
    linewidths=2.5,
    label="Observed cumulative through May 1, 2026",
)

ax.set_xlim(fitted_values.index.min(), FULL_YEAR_2026_FORECAST_DATE)
ax.set_ylim(bottom=50000)
ax.set_title("Cumulative measles cases: monotone AutoARIMAX fit and 2026 forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative cases")
ax.legend()
plt.tight_layout()
plt.savefig("autoarimax_cumulative_forecast.png", dpi=300)
plt.show()

model_summary.to_csv("autoarimax_model_summary.csv", index=False)
param_uncertainty.to_csv("autoarimax_parameter_uncertainty.csv", index_label="parameter")
fitted_values.to_csv("autoarimax_fitted_values.csv")

print("Saved autoarimax_model_summary.csv")
print("Saved autoarimax_parameter_uncertainty.csv")
print("Saved autoarimax_fitted_values.csv")
print("Saved autoarimax_cumulative_forecast.png")


Saved autoarimax_model_summary.csv
Saved autoarimax_parameter_uncertainty.csv
Saved autoarimax_fitted_values.csv
Saved autoarimax_cumulative_forecast.png


## Terminal Review

```bash
column -s, -t < autoarimax_model_summary.csv
column -s, -t < autoarimax_parameter_uncertainty.csv
```
